In [1]:
!pip install -q whisperx
!pip install -q soundfile

In [2]:
import os
import json
import gc
from datetime import datetime, timezone

import torch
import whisperx

print("Torch:", torch.__version__, "| CUDA disponível:", torch.cuda.is_available())

Torch: 2.8.0+cu128 | CUDA disponível: True


In [3]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
CONFIG = {
    # --- Caminhos no Drive ---
    "base_dir": "/content/drive/MyDrive/Arquivos gerados do projeto",
    "audio_relpath": "áudios/primeiraruiniao.mp4",
    "output_relpath": "transcrições/primeirareuniao.json",

    # --- Modelo WhisperX ---
    "model_size": "turbo",       # tiny/base/small/medium/large-v2/large-v3
    "language": "pt",               # None para detecção automática
    "batch_size": 16,               # reduzir se faltar VRAM
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "compute_type": "float16" if torch.cuda.is_available() else "int8",

    # --- Áudio (pensado para evolução futura) ---
    "expected_sample_rate": 16000,
    "assume_preprocessed_audio": False,
}

AUDIO_PATH = os.path.join(CONFIG["base_dir"], CONFIG["audio_relpath"])
OUTPUT_PATH = os.path.join(CONFIG["base_dir"], CONFIG["output_relpath"])

print("Áudio de entrada :", AUDIO_PATH)
print("JSON de saída    :", OUTPUT_PATH)
print("Device / compute :", CONFIG["device"], "/", CONFIG["compute_type"])

Áudio de entrada : /content/drive/MyDrive/Arquivos gerados do projeto/áudios/primeiraruiniao.mp4
JSON de saída    : /content/drive/MyDrive/Arquivos gerados do projeto/transcrições/primeirareuniao.json
Device / compute : cuda / float16


In [5]:
import soundfile as sf


def load_and_prepare_audio(audio_path: str, config: dict):
    '''Carrega o áudio e devolve o array de amostras (float32, mono, 16 kHz).

    Isolar essa lógica aqui permite trocar a estratégia de carregamento
    sem alterar as células de transcrição/alinhamento abaixo.
    '''
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Áudio não encontrado em: {audio_path}")

    expected_sr = config["expected_sample_rate"]

    if config["assume_preprocessed_audio"]:
        # Cenário futuro: o áudio já deve chegar pronto (16 kHz, mono).
        info = sf.info(audio_path)
        if info.samplerate != expected_sr or info.channels != 1:
            raise ValueError(
                f"Áudio fora do padrão esperado ({expected_sr} Hz, mono). "
                f"Recebido: {info.samplerate} Hz, {info.channels} canal(is)."
            )
        audio = whisperx.load_audio(audio_path)
    else:
        # Cenário atual: aceita qualquer formato/sample rate (ex.: .mp4);
        # o WhisperX reamostra internamente para 16 kHz mono.
        audio = whisperx.load_audio(audio_path)

    return audio


audio = load_and_prepare_audio(AUDIO_PATH, CONFIG)
duration_seconds = len(audio) / 16000
print(f"Áudio carregado — duração aproximada: {duration_seconds:.1f}s")

Áudio carregado — duração aproximada: 2038.6s


In [6]:
model = whisperx.load_model(
    CONFIG["model_size"],
    device=CONFIG["device"],
    compute_type=CONFIG["compute_type"],
    language=CONFIG["language"],
)

result = model.transcribe(audio, batch_size=CONFIG["batch_size"])

detected_language = result.get("language", CONFIG["language"])
print(f"Idioma detectado/usado: {detected_language}")
print(f"Segmentos brutos: {len(result['segments'])}")

# Libera memória do modelo de transcrição antes de carregar o de alinhamento
del model
gc.collect()
if CONFIG["device"] == "cuda":
    torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

vocabulary.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

2026-08-01 04:09:23 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issue

Idioma detectado/usado: pt
Segmentos brutos: 65


In [7]:
align_model, align_metadata = whisperx.load_align_model(
    language_code=detected_language,
    device=CONFIG["device"],
)

result_aligned = whisperx.align(
    result["segments"],
    align_model,
    align_metadata,
    audio,
    CONFIG["device"],
    return_char_alignments=False,
)

del align_model
gc.collect()
if CONFIG["device"] == "cuda":
    torch.cuda.empty_cache()

print(f"Segmentos alinhados: {len(result_aligned['segments'])}")

preprocessor_config.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Segmentos alinhados: 325


In [8]:
def build_output_json(result_aligned: dict, config: dict, audio_path: str,
                       duration_seconds: float, detected_language: str) -> dict:
    segments = []
    for i, seg in enumerate(result_aligned["segments"]):
        words = [
            {
                "word": w.get("word"),
                "start": w.get("start"),
                "end": w.get("end"),
                "score": w.get("score"),
            }
            for w in seg.get("words", [])
            # descarta palavras sem timestamp (ex.: falhas pontuais de alinhamento)
            if w.get("start") is not None and w.get("end") is not None
        ]
        segments.append({
            "id": i,
            "start": seg.get("start"),
            "end": seg.get("end"),
            "text": seg.get("text", "").strip(),
            "words": words,
        })

    output = {
        "metadata": {
            "audio_file": os.path.basename(audio_path),
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "pipeline_stage": "transcricao",
            "model": config["model_size"],
            "language": detected_language,
            "duration_seconds": round(duration_seconds, 3),
            "sample_rate": config["expected_sample_rate"],
        },
        "segments": segments,
    }
    return output


output_json = build_output_json(
    result_aligned, CONFIG, AUDIO_PATH, duration_seconds, detected_language
)

print(json.dumps(output_json["metadata"], indent=2, ensure_ascii=False))

{
  "audio_file": "primeiraruiniao.mp4",
  "generated_at": "2026-08-01T04:11:02.487181+00:00",
  "pipeline_stage": "transcricao",
  "model": "turbo",
  "language": "pt",
  "duration_seconds": 2038.592,
  "sample_rate": 16000
}


In [9]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output_json, f, ensure_ascii=False, indent=2)

print(f"JSON salvo em: {OUTPUT_PATH}")

JSON salvo em: /content/drive/MyDrive/Arquivos gerados do projeto/transcrições/primeirareuniao.json


In [10]:
# Descomente e ajuste antes de usar.

# from google.colab import userdata
#
# GH_TOKEN = userdata.get("GH_TOKEN")
# REPO_URL = f"https://{GH_TOKEN}@github.com/<usuario>/<repositorio>.git"
# BRANCH = "feature/whisper-transcription"
#
# %cd /content
# !git clone -b {BRANCH} {REPO_URL} repo 2>/dev/null || (cd repo && git checkout {BRANCH})
# !cp "01_transcricao.ipynb" repo/notebooks/ 2>/dev/null
# %cd repo
# !git add notebooks/01_transcricao.ipynb
# !git commit -m "feat(transcricao): atualiza notebook de transcrição WhisperX"
# !git push origin {BRANCH}